# 02_clean_prices.py 결과 확인

`data/cleaned/prices`의 결측 보간(`is_interpolated`) / 액면분할 의심 탐지(`split_suspected`) 결과를 확인한다.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.master("spark://spark-master:7077").appName("check_cleaned_prices").getOrCreate()

df = spark.read.parquet("/opt/spark-apps/data/cleaned/prices")
print(f"전체 {df.count()}건")
df.groupBy("snapshot_type").count().toPandas()

전체 35692건


,snapshot_type,count
0,current,30650
1,12m_ago,2492
2,1m_ago,2550


## 1. 보간된 행 (`is_interpolated=true`)
직전 종가로 채운 결측 거래일. 거래정지 구간도 원본에 시세가 없는 형태로 들어오므로 여기 포함된다.

In [3]:
interpolated = df.filter(F.col("is_interpolated"))
print(f"보간된 행: {interpolated.count()}건")
interpolated.select("stock_code", "bas_dt", "close_price", "is_interpolated") \
    .orderBy("stock_code", "bas_dt") \
    .toPandas()

보간된 행: 0건


,stock_code,bas_dt,close_price,is_interpolated


## 2. 액면분할/병합 의심 행 (`split_suspected=true`)
전일 대비 종가 등락률이 -40% 이하 또는 +67% 이상인 지점. 자동 보정 없이 플래그만 표시됨 — 진짜 분할/병합인지, 단순 급등락인지는 별도 확인 필요.

In [4]:
w = Window.partitionBy("stock_code").orderBy("bas_dt")
suspects = df.withColumn("prev_close", F.lag("close_price").over(w)) \
    .filter(F.col("split_suspected")) \
    .withColumn("day_over_day_rate", F.round((F.col("close_price") - F.col("prev_close")) / F.col("prev_close") * 100, 2))

print(f"액면분할/병합 의심 행: {suspects.count()}건")
suspects.select("stock_code", "bas_dt", "prev_close", "close_price", "day_over_day_rate") \
    .orderBy("stock_code", "bas_dt") \
    .toPandas()

액면분할/병합 의심 행: 24건


,stock_code,bas_dt,prev_close,close_price,day_over_day_rate
0,000040,20260728,267,1047,292.13
1,001520,20260720,550,995,80.91
2,004870,20260724,247,979,296.36
3,006490,20260716,310,1177,279.68
4,007720,20260714,299,1456,386.96
5,008470,20260722,3360,1730,-48.51
6,009730,20260720,181,1726,853.59
7,011000,20260721,534,2670,400.00
8,013720,20260728,431,1750,306.03
9,025560,20260727,41750,10210,-75.54


## 3. 정제 후에도 남은 결측치 (컬럼별 null 개수)
전부 0이어야 정상.

In [5]:
null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
null_counts.toPandas()

,stock_code,bas_dt,close_price,fluctuation_rate,listed_share_count,market_cap,open_price,snapshot_type,is_interpolated,split_suspected,year
0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
spark.stop()